# YouTube 字幕とコメントの感情分析（セグメントごと）
このノートブックでは、YouTube動画の特定の時間セグメントごとに、字幕およびリアルタイムコメントの感情（Positive/Neutral/Negative）の比率を算出して集計します。

**使用モデル**: `LoneWolfgang/bert-for-japanese-twitter-sentiment`
- 日本語Twitter特化のBERTモデル
- ラベル: `negative` (0) / `neutral` (1) / `positive` (2)

In [ ]:
# 1. MeCabおよび辞書のインストール（Google Colab環境用）
!apt-get -q -y install mecab libmecab-dev mecab-ipadic-utf8
!pip install mecab-python3 ipadic fugashi unidic-lite --quiet

# パス不一致を解消するためのシンボリックリンク作成
!ln -s /etc/mecabrc /usr/local/etc/mecabrc

# 2. その他の必要なライブラリのインストール
!pip install transformers sentencepiece pytchat youtube-transcript-api pytube pandas numpy --quiet

In [ ]:
# 3. 特定のブランチを指定してクローン
!git clone -b refactor/memory-improvements https://github.com/ShotaMiwa/year1-research.git /content/year1

# 4. モジュールインポートのパスを通す
import sys
sys.path.append('/content/year1/src')
sys.path.append('/content/year1/src/data_creaters/簡易化版')
sys.path.append('/content/year1/googlecolab')

In [ ]:
# 5. 感情分析用スクリプトの読み込みと実行
import torch
from sentiment_segment_analysis import parse_timetable, load_sentiment_pipeline, analyze_sentiment_by_segments

# --- 実行設定 ---
VIDEO_URL = "https://www.youtube.com/watch?v=pP2KLW-_7hQ"
TIMETABLE_RAW = """
0:01:49 0:03:44
0:03:46 0:06:00
0:06:56 0:08:47
0:08:47 0:09:51
0:10:07 0:11:49
0:11:50 0:14:22
0:14:24 0:15:30
0:16:10 0:17:36
0:17:36 0:18:38
0:20:53 0:21:57
0:23:47 0:26:08
0:28:34 0:32:21
0:32:41 0:33:48
0:34:30 0:36:42
0:39:13 0:40:37
0:45:51 0:47:08
0:52:30 0:54:47
0:55:17 0:57:42
0:57:44 0:58:44
1:00:35 1:03:03
1:03:06 1:04:24
1:04:47 1:06:43
1:10:12 1:14:51
"""

# タイムテーブルパースとモデルのロード
segments = parse_timetable(TIMETABLE_RAW)
device = 0 if torch.cuda.is_available() else -1
sentiment_pipeline = load_sentiment_pipeline(device=device)

# 感情比率の計算
df_results = analyze_sentiment_by_segments(
    video_url=VIDEO_URL,
    segments=segments,
    sentiment_pipeline=sentiment_pipeline,
    enable_comments=True
)

# 結果の表示
display(df_results)
